# About the Notebook


## 📰 Search with LLMs: Recent News Summarizer

In this example, we build a streamlined **news summarization app** powered by LLMs and the [Exa API](https://exa.ai). The system works by:

- Using Exa to **search for recent and relevant links** on a given topic;
- **Extracting clean content** directly from those links;
- Summarizing the information with an LLM to provide a concise and current answer.

This approach ensures your responses are based on **fresh, verifiable sources** — no need for custom web scraping or crawling.

You can try it directly in this [Colab notebook](https://colab.research.google.com/drive/1w1WaPpbdm8fAPw_B5M0U-GHIZpcIbUaw), or explore the [GitHub repo](https://github.com/exa-labs/exa-py/tree/master/examples/newssummarizer/summarizer.ipynb) for both the notebook and a [pure Python version](https://github.com/exa-labs/exa-py/tree/master/examples/newssummarizer/summarizer.py).

To get started, you’ll need API keys from both [Exa](https://dashboard.exa.ai/overview) and [OpenAI](https://platform.openai.com/api-keys).

# Dependencies

In [ ]:
%%capture
!pip install openai==1.61.1 python-dotenv==1.0.1 exa_py==1.7.0

# API Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set up API keys and environment variables
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
EXA_API_KEY = os.getenv('EXA_API_KEY')


# Constants

In [ ]:
model="gpt-4o-mini"

# Imports

In [ ]:
import textwrap
import openai
from exa_py import Exa
from datetime import datetime, timedelta

# Retrieving news with Exa
Let's use the Exa neural search engine to search the web for relevant links to the user's question.

First, we ask the LLM to generate a search engine query based on the question.

In [ ]:
openai.api_key = OPENAI_API_KEY
exa = Exa(EXA_API_KEY)

SYSTEM_MESSAGE = "You are a helpful assistant that generates search queries based on user questions. Only generate one search query."
USER_QUESTION = "Gold"

completion = openai.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_QUESTION},
    ],
)

search_query = completion.choices[0].message.content

print("Search query:")
print(search_query)

Search query:
What are the current prices and trends in the gold market?


Now let's put the search query into Exa. Let's also use `start_published_date` to filter the results to pages published in the last week. Notice that we set `use_autoprompt=True` which lets the Exa API further optimize our search query for best results. Essentially, there is a special way to format Exa queries for best results, which `autoprompt` does automatically

In [ ]:
one_week_ago = (datetime.now() - timedelta(days=7))
date_cutoff = one_week_ago.strftime("%Y-%m-%d")

search_response = exa.search_and_contents(
    search_query, use_autoprompt=True, start_published_date=date_cutoff
)

urls = [result.url for result in search_response.results]
print("URLs:")
for url in urls:
    print(url)

URLs:
https://www.reuters.com/markets/commodities/gold-prices-ease-record-highs-trump-grants-tariff-exemptions-2025-04-14/
https://www.cbsnews.com/news/gold-price-scenarios-that-could-happen-april-2025-experts/
https://finance.yahoo.com/personal-finance/investing/article/gold-price-today-monday-april-14-2025-gold-opens-strong-133259858.html
https://www.reuters.com/markets/commodities/goldman-sachs-raises-end-2025-gold-price-forecast-3700oz-2025-04-14/
https://finance.yahoo.com/personal-finance/investing/article/gold-price-today-thursday-april-17-2025-golds-strength-continues-131157963.html
https://www.cbsnews.com/news/gold-prices-soar-past-3300-does-it-still-make-sense-to-invest-now/
https://www.cnbc.com/2025/04/16/safe-haven-gold-hits-record-high-on-weaker-dollar-trade-war-concerns.html
https://markets.businessinsider.com/news/commodities/gold-price-target-goldman-sachs-safe-haven-assets-tariffs-markets-2025-4
https://finance.yahoo.com/news/why-gold-prices-surged-record-192553739.html

Exa returned a curated list of highly relevant URLs based on our original question — not just links, but **useful content** too.

You might be wondering: *How is this different from using Google?* Good question. Try [Googling something like “Recent breakthroughs in physics news”](https://www.google.com/search?q=Recent+breakthroughs+in+physics+news). You’ll mostly get front pages of news sites or aggregator platforms — not the articles themselves.

With Exa’s `search_and_contents` feature, you get both the **links and the actual webpage content** in one go. That means you can skip building a web crawler and immediately start working with the real information.

In [ ]:
results = search_response.results
result_item = results[0]
print(f"{len(results)} items total, printing the first one:")
print(result_item.text)

10 items total, printing the first one:
A woman looks at a gold bangle inside a jewellery showroom at a market in Mumbai January 15, 2015. REUTERS/Shailesh Andrade/File Photo Purchase Licensing Rights , opens new tab Summary Gold retreats from record high of $3,245.42 Goldman Sachs raises end-2025 gold price forecast to $3,700/oz Palladium up more than 4% April 14 (Reuters) - Gold prices dipped on Monday, retreating from a record high hit earlier in the day, as risk appetite improved after the White House exempted smartphones and computers from steep tariffs on China. Spot gold was down 0.7% at $3,213.69 an ounce, as of 01:52 p.m. ET (1752 GMT), after hitting an all-time high of $3,245.42. U.S. gold futures settled 0.6% lower at $3,226.30. Sign up here. "Some risk-on trading here got us off the recent highs, but still the environment is pretty good for gold," said Bart Melek, head of commodity strategies at TD Securities. Risk sentiment in wider financial markets ticked higher after Wa

Perfect, we’ve pulled in 10 results and taken a quick look at the first one. But let’s be honest, we’re not about to read through them all manually. Instead, let’s have the LLM do the heavy lifting and summarize the content for us.

In [ ]:
SYSTEM_MESSAGE = "You are a helpful assistant that briefly summarizes the content of a webpage. Summarize the users input."

completion = openai.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": result_item.text},
    ],
)

summary = completion.choices[0].message.content

print(f"Summary for {urls[0]}:")
print(result_item.title)
print(textwrap.fill(summary, 80))

Summary for https://www.reuters.com/markets/commodities/gold-prices-ease-record-highs-trump-grants-tariff-exemptions-2025-04-14/:
Gold takes a breather after record run as risk sentiment improves
Gold prices retreated from a record high of $3,245.42 due to improved risk
appetite following the U.S. government's exemption of certain electronics from
tariffs on China. Spot gold fell by 0.7% to $3,213.69 an ounce, while U.S. gold
futures declined by 0.6% to $3,226.30. Analysts noted that while increased risk
sentiment reduced safe-haven demand for gold, ongoing trade uncertainties and a
weak dollar continue to support prices. Goldman Sachs has raised its end-2025
gold price forecast to $3,700, driven by strong central bank demand and
recession risks. Additionally, investment flows into Chinese gold ETFs have
exceeded those in U.S. funds this month. Other precious metals like silver,
platinum, and palladium saw slight increases.
